# Week 6 - 02: Store Embeddings in a Vector Database

## Goal
We will:
1. Install/use ChromaDB.
2. Generate embeddings for document chunks.
3. Store the chunks and embeddings in ChromaDB.
4. Save the database locally.

We will use:
- `sentence-transformers` -> creates embeddings locally
- `chromadb` -> stores and searches vectors

No API key is required for this notebook.


In [ ]:
# The ! means "run this command in the system terminal".
%pip install chromadb sentence-transformers

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [4]:
# Create a local ChromaDB database.
# PersistentClient means the data is saved on your computer.
# "./chroma_week6" is the folder where Chroma stores the database.

client = chromadb.PersistentClient(path="./chroma_week6")
print("ChromaDB client created!")

ChromaDB client created!


In [6]:
# get_or_create_collection() means:
# - create it if it does not exist
# - use the existing one if it already exists

collection = client.get_or_create_collection(name="week6_documents")
print("Collection created!")

Collection created!


In [8]:
documents = [
    "Artificial Intelligence allows computers to perform tasks that normally require human intelligence.",
    "Machine learning is a part of AI where computers learn patterns from data.",
    "Natural language processing helps computers work with human language such as text and speech.",
    "Vector databases store embeddings and make similarity search fast.",
    "Semantic search finds information based on meaning, not only exact keywords.",
    "A timetable scheduling system can use AI to assign teachers, rooms, courses, and time slots.",
    "Constraint satisfaction problems are useful when a scheduling problem has many rules.",
    "Genetic algorithms can search for good solutions by using selection, crossover, and mutation."
]
print("Number of documents:", len(documents))

Number of documents: 8


In [9]:
# Convert every document into an embedding.
# encode() returns a numerical vector for each text.
# convert_to_numpy=True makes the result easy to use with Python.

embeddings = model.encode(documents, convert_to_numpy=True)

print("Number of embeddings:", len(embeddings))
print("Size of one embedding:", len(embeddings[0]))
print("First few values of first embedding:", embeddings[0][:5])


Number of embeddings: 8
Size of one embedding: 384
First few values of first embedding: [-0.02316255  0.03050832  0.06770162  0.00595455 -0.01488171]


In [10]:
# Give every document a unique ID.
ids = [f"doc_{i}" for i in range(len(documents))]

# Add documents + embeddings to ChromaDB.
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()
)

print("Documents and embeddings stored successfully!")
print("Total items in collection:", collection.count())


Documents and embeddings stored successfully!
Total items in collection: 8


In [11]:
# Test that our data is really inside the vector database.

stored_data = collection.get()

print("IDs:", stored_data["ids"])
print("\nFirst document:", stored_data["documents"][0])


IDs: ['doc_0', 'doc_1', 'doc_2', 'doc_3', 'doc_4', 'doc_5', 'doc_6', 'doc_7']

First document: Artificial Intelligence allows computers to perform tasks that normally require human intelligence.


## Important idea

The vector database does not simply remember:

`"AI is a field..."`

It also stores the numerical embedding of that text.

Later, when we ask a question:
1. The question is converted into an embedding.
2. Chroma compares that vector with stored vectors.
3. The closest vectors are returned.
4. We get the original text belonging to those vectors.

That is the basic idea of semantic search.
